# Are the rules any good?

`categorize.coverage()` can tell you how much of a statement the keyword
rules matched. It cannot tell you whether a match was **correct**, because
a bank statement comes with no correct answer attached.

This notebook borrows one that does: 2,461 transactions a real person
categorised by hand while tracking their own spending. With ground truth
available, three approaches can be scored on identical data.

Run order matters — each cell depends on the one above.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from expense_analyzer import benchmark, evaluate, model

pd.set_option("display.max_colwidth", 60)

## 1. Build the benchmark

Downloads on first run, then caches. Three preparation choices are made
here, and all three change the numbers — see `docs/decisions/0005`.

In [2]:
bench = benchmark.load_benchmark()
print(bench.summary())
print()
print("classes:", ", ".join(bench.classes))

C:\Users\tdmne\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1,940 labelled rows over 12 classes (1,455 train / 485 test)
521 rows dropped for having no note
120 rows moved to 'Other' (class had under 20 examples)

classes: Apparel, Beauty, Family, Food, Gift, Health, Household, Investment, Other, Salary, Transportation, subscription


In [3]:
# What the text actually looks like. This is the whole story in one cell:
# these are notes a human wrote for themselves, not bank narrations.
bench.train[["narration", "label"]].head(12)

,narration,label
0,Toast packet,Food
1,Home Food Delivery,Food
2,2 Current Residence to Place 0,Transportation
3,grocery,Food
4,rakshabandhan ovalni,Other
5,2 cutting chai,Food
6,1 lit,Food
7,onion 2 kg,Food
8,half liter buffalo,Food
9,1.5 lit milk,Food


## 2. The floor

Before anything clever, establish what "doing nothing" scores. Food is 43%
of the data, so a model that always answers Food gets 43% accuracy — and
has learned nothing at all. That gap between accuracy and macro F1 is why
both are reported everywhere in this project.

In [4]:
truth = bench.test["label"]

majority = evaluate.score(
    "majority class", truth, benchmark.predict_majority(bench.train, bench.test)
)
print(f"accuracy {majority.accuracy:.1%}   macro F1 {majority.macro_f1:.3f}")

accuracy 42.9%   macro F1 0.050


## 3. The keyword rules

The same `categorize()` the main pipeline uses, translated into this
dataset's category names. The translation is deliberately generous — a
baseline is only meaningful if it is as strong as it honestly can be.

In [5]:
rules = evaluate.score("keyword rules", truth, benchmark.predict_rules(bench.test))
print(f"accuracy {rules.accuracy:.1%}   macro F1 {rules.macro_f1:.3f}")

accuracy 13.4%   macro F1 0.035


That is **worse than doing nothing**. Worth understanding before moving on —
look at what the rules actually produce.

In [6]:
from expense_analyzer.categorize import UNCATEGORIZED, categorize

raw_rule_output = bench.test["narration"].apply(categorize)
print(raw_rule_output.value_counts().to_string())
print(f"\nuncategorised: {(raw_rule_output == UNCATEGORIZED).mean():.1%}")
print("\nnotes the rules cannot touch:")
for note in bench.test.loc[raw_rule_output == UNCATEGORIZED, "narration"].head(8):
    print("   ", note[:60])

narration
Uncategorized    452
Rent              18
Bills             10
Food               3
Shopping           1
Groceries          1

uncategorised: 93.2%

notes the rules cannot touch:
    From workplace
    internet renewal (3 months pack)
    fruits and vegetables
    Jel
    Mutual fund A
    grocery
    Shengdane pav kg
    2 Place 0 to WS single


The rules encode **merchant names**. These notes contain **descriptions of
things** — *fruits and vegetables*, *Shengdane pav kg*. There is no overlap
to match on, so 93% falls through.

This is not evidence that rules are bad. It is evidence that they do not
transfer between two different kinds of text. On their own domain they are
precise and completely explainable.

## 4. A classifier that learns this vocabulary

TF-IDF into logistic regression. Word n-grams catch *grocery*; character
n-grams catch the typos and transliteration that word n-grams miss.

In [7]:
pipeline = model.train(bench.train)
predictions = model.predict(pipeline, bench.test)

learned = evaluate.score("TF-IDF + LogReg", truth, predictions)
print(evaluate.comparison_table([majority, rules, learned]))

approach                   accuracy   macro F1  weighted F1       n
-------------------------------------------------------------------
majority class                42.9%      0.050        0.257     485
keyword rules                 13.4%      0.035        0.046     485
TF-IDF + LogReg               87.0%      0.769        0.864     485


## 5. Is that real, or a lucky split?

One test set can flatter a model. Cross-validation refits on five different
splits of the training data alone. A tight spread close to the test score
means the result holds.

In [8]:
from sklearn.model_selection import cross_val_score

cv = cross_val_score(
    model.build_pipeline(),
    bench.train["narration"],
    bench.train["label"],
    cv=5,
    scoring="f1_macro",
)
print(f"macro F1 across 5 folds: {cv.mean():.3f} +/- {cv.std():.3f}")
print(f"held-out test macro F1:  {learned.macro_f1:.3f}")

macro F1 across 5 folds: 0.740 +/- 0.034
held-out test macro F1:  0.769


## 6. Where it goes wrong

An average hides everything interesting. These are the classes it misses
and the specific mistakes it makes.

In [9]:
evaluate.per_class_report(truth, predictions)

,precision,recall,f1-score,support
Family,0.000,0.000,0.000,6
Gift,0.800,0.571,0.667,7
Household,0.684,0.634,0.658,41
Other,0.717,0.705,0.711,61
Health,0.842,0.727,0.780,22
Apparel,0.625,0.833,0.714,12
subscription,0.913,0.913,0.913,23
Food,0.930,0.952,0.941,208
Transportation,0.949,0.987,0.967,75
Beauty,0.833,1.000,0.909,5


In [10]:
evaluate.worst_confusions(truth, predictions)

,actual,predicted,count
0,Household,Food,8
1,Food,Household,6
2,Household,Other,5
3,Other,Household,4
4,Other,Apparel,3
5,Food,Other,3
6,Health,Other,3
7,Other,Transportation,3
8,subscription,Other,2
9,Other,Food,2


`Household` and `Food` trade places in both directions — a genuinely hard
boundary, since groceries bought to cook with and a ready-made meal use the
same words. `Family` is never predicted at all: 23 examples, and no
vocabulary of its own.

## 7. Check it learned something sensible

A score is not enough. Read the words the model leans on — a model relying
on something absurd has memorised a quirk of this dataset rather than
learning a category. This is the main reason for choosing a model whose
weights can be read at all.

In [11]:
for label in ["Transportation", "Health", "Investment"]:
    top = model.top_features_for(pipeline, label, top=8)
    print(f"{label}: {', '.join(top['feature'])}")
    print()

Transportation: word__to, word__place, char__to , char__o , char__ to , char__ to, char__to, word__place to

Health: word__glasses, word__tablet, word__medicine, word__strepsils, word__eye, word__drop, word__consultation, word__hospital consultation

Investment: word__mutual, word__mutual fund, word__fund, word__rd, word__icici, word__icici prudential, word__prudential, word__insurance



## What to take away

1. **Measure the simple thing first.** Without the 42.9% floor, 87% would
   be an unfalsifiable number.
2. **Accuracy alone misleads on imbalanced data.** Always-Food scores 42.9%
   accuracy and 0.050 macro F1. The second number is the honest one.
3. **A model is not automatically better than rules.** Here it is, on this
   text. On bank narrations the rules are precise and explainable, and the
   13.4% measures transfer between domains — not quality.
4. **Read the errors, not just the average.** `Family` at zero recall is
   invisible in the headline figure.

Full write-up: [`docs/results.md`](../docs/results.md).